<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

<font size="5" color='grey'> <b>
Advanced RAG – Pipeline-Patterns
</b></font> </br>

---

**Beitrag zum Leitprojekt:** Advanced RAG verbessert das Evidence Tool des Meeting- & Briefing-Agent dort, wo Basic RAG an Grenzen stößt — Reranking, Self-RAG und Corrective RAG erhöhen die Trefferqualität, ändern aber nichts an der Grundregel: Ohne belegbaren Treffer keine Behauptung.

**Praxisanker:** Mara Vogt profitiert von verbessertem Retrieval, wenn ein langes Projektarchiv relevante Passagen, Metadaten und widersprüchliche Stände zuverlässig liefern soll.


In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "m28-advanced-rag"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "tags": ["m28", "rag"],
    "metadata": {"notebook": "m28", "version": "1.0"}
}


✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 35.201.172.19
Hostname: 19.172.201.35.bc.googleusercontent.com
Stadt: Taipei
Region: Taiwan
Land: TW
Koordinaten: 25.0531,121.5264
Provider: AS396982 Google LLC
Postleitzahl: None
Zeitzone: Asia/Taipei


In [2]:
#@title 📦 Installationen { display-mode: "form" }
install_packages([
    ('pypdf', 'pypdf'),
])


🔄 Installiere pypdf...
✅ pypdf erfolgreich installiert und importiert


In [ ]:
#@title 📂 Meeting-Briefing-Korpus laden { display-mode: "form" }
import shutil
from genai_lib.utilities import copy_from_github

KORPUS_QUELLE = "ralf-42/Agenten/02_daten/01_text/korpus_meeting_briefing"
KORPUS_MASKE = "*.pdf"
KORPUS_TARGET = "files"

shutil.rmtree(KORPUS_TARGET, ignore_errors=True)
copy_from_github(
    source=KORPUS_QUELLE,
    target=KORPUS_TARGET,
    mask=KORPUS_MASKE,
)


# 1 | Von Basic zu Advanced RAG
---

<p><font color='black' size='5'>
Motivation
</font></p>

In ***RAG-Konzepte & Embeddings*** wurden die Grundlagen von RAG (Retrieval-Augmented Generation) eingeführt:

```
User Query → Retrieve Documents → Generate Answer
```

**Das funktioniert gut, aber es gibt Probleme:**

❌ **Problem 1: Irrelevante Dokumente**
- Der Retriever findet manchmal unpassende Dokumente
- Diese verwirren das LLM und führen zu schlechten Antworten

❌ **Problem 2: Halluzinationen**
- Das LLM erfindet Fakten, die nicht in den Dokumenten stehen
- Keine Überprüfung der generierten Antwort

❌ **Problem 3: Unflexibel**
- Der Workflow ist statisch: Immer retrieve → generate
- Keine Anpassung an die Qualität der Zwischenergebnisse

**Lösung: Advanced RAG mit LangGraph!**

✅ **Self-RAG:** Bewertet gefundene Dokumente vor der Generierung    
✅ **Corrective RAG:** Verbessert Queries bei schlechten Ergebnissen   
✅ **Adaptive RAG:** Passt den Workflow dynamisch an   

<p><font color='black' size='5'>
Übersicht der Patterns
</font></p>

| Pattern | Was macht es? | Wann verwenden? |
|---------|---------------|------------------|
| **Self-RAG** | Bewertet Relevanz der Dokumente | Wenn Retrieval-Qualität wichtig ist |
| **Reranking** | Sortiert Docs nach Relevanz-Score | Wenn Reihenfolge der Docs wichtig ist |
| **Corrective RAG** | Verbessert Query bei schlechten Retrievals | Wenn Nutzer-Queries ungenau sind |
| **Adaptive RAG** | Wählt beste Strategie basierend auf Query-Typ | Für vielseitige Anwendungen |

**In diesem Modul bauen wir:**
1. Self-RAG mit Document Grading
2. Reranking: Dokumente nach Relevanz-Score sortieren
3. Corrective RAG mit Query Rewriting
4. Adaptive RAG als Abschlussaufgabe (Routing zwischen den Patterns)

# 2 | Setup & Vorbereitung
---

In [ ]:
# Imports
from typing import Literal, TypedDict, List
from pathlib import Path
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

# LangGraph
from langgraph.graph import StateGraph, START, END

In [ ]:
from genai_lib.model_config import WORKER
# ── Konfigurationskonstanten ───────────────────────────────────────────────
MAX_RETRIES  = 3   # API-Retry-Versuche bei transienten Fehlern (with_retry)
K_RETRIEVAL  = 3   # Standard-Anzahl abgerufener Dokumente
K_WIDE       = 6   # Breites Retrieval für Reranking (mehr Kandidaten)

# Modell und Embeddings
llm = init_chat_model(WORKER)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


<p><font color='black' size='5'>
Meeting-Briefing-PDFs laden
</font></p>

Der gemeinsame Meeting-Briefing-Korpus wird aus PDF-Dateien geladen und in eine Wissensbasis für Advanced RAG überführt.


In [ ]:
# PDF-Dokumente laden: pypdf.PdfReader + einfaches Chunking
from pypdf import PdfReader
from uuid import uuid4


def einfache_chunks(text: str, chunk_size: int = 1000, overlap: int = 200) -> list[str]:
    """Teilt Text ohne zusätzliche Splitter-Abhängigkeit in überlappende Chunks."""
    sauber = " ".join(text.split())
    if not sauber:
        return []
    step = max(1, chunk_size - overlap)
    return [sauber[i:i + chunk_size] for i in range(0, len(sauber), step)]


def add_documents_batched(store: Chroma, docs: list[Document], batch_size: int = 96) -> None:
    """Fügt Chroma-Dokumente in kleinen Batches hinzu."""
    for start in range(0, len(docs), batch_size):
        store.add_documents(docs[start:start + batch_size])


files_dir = Path(KORPUS_TARGET)
pdf_paths = sorted(files_dir.rglob("*.pdf"))

documents = []
split_docs = []

for pdf_path in pdf_paths:
    try:
        reader = PdfReader(str(pdf_path))
        pages = [page.extract_text() or "" for page in reader.pages]
        text = "\n".join(pages)
        documents.append(Document(page_content=text, metadata={"source": pdf_path.name}))
        for chunk_id, chunk in enumerate(einfache_chunks(text)):
            split_docs.append(Document(
                page_content=chunk,
                metadata={"source": pdf_path.name, "chunk_id": chunk_id},
            ))
        print(f"✅ Geladen: {pdf_path.name} ({len(pages)} Seiten)")
    except Exception as e:
        print(f"⚠️ Fehler bei {pdf_path.name}: {e}")

print(f"\n📄 {len(documents)} PDFs geladen")
print(f"📦 {len(split_docs)} Chunks erstellt")

vectorstore = Chroma(
    collection_name=f"advanced_rag_briefing_{uuid4().hex[:8]}",
    embedding_function=embeddings,
)
add_documents_batched(vectorstore, split_docs)

# Retriever: K_RETRIEVAL als benannte Konstante statt Magic Number
retriever = vectorstore.as_retriever(search_kwargs={"k": K_RETRIEVAL})

print(f"✅ Vektordatenbank mit {len(split_docs)} Chunks erstellt")
print(f"   Retriever: k={K_RETRIEVAL} (Standard), k={K_WIDE} (Wide/Reranking)")

# 3 | Self-RAG: Document Grading
---

<p><font color='black' size='5'>
Konzept
</font></p>

**Self-RAG fügt einen Bewertungsschritt hinzu:**

```
Query → Retrieve → [GRADE DOCS] → Generate (nur mit relevanten Docs)
```

**Warum wichtig?**
- Irrelevante Dokumente werden herausgefiltert
- Das LLM bekommt nur hochwertige Informationen
- Reduziert Halluzinationen und verbessert Genauigkeit

<p><font color='black' size='5'>
State Definition
</font></p>

In [7]:
class SelfRAGState(TypedDict):
    """State für Self-RAG Workflow"""
    query: str                          # Nutzer-Anfrage
    documents: List[Document]           # Abgerufene Dokumente
    relevant_docs: List[Document]  # Gefilterte relevante Dokumente
    answer: str                         # Generierte Antwort
    relevance: List[str]         # "yes"/"no" für jedes Dokument

<p><font color='black' size='5'>
Document Grader mit structured output
</font></p>

In [8]:
# Pydantic-Modell für Bewertung
class GradeDocumentsResult(BaseModel):
    """Bewertet ob ein Dokument relevant für die Query ist"""
    score: str = Field(
        description="'yes' wenn relevant, 'no' wenn nicht relevant"
    )

# .with_retry(): schützt vor transienten API-Fehlern beim Grading
grader_llm = llm.with_structured_output(GradeDocumentsResult).with_retry(stop_after_attempt=MAX_RETRIES)

# Grading Prompt – streng: nur direkte Relevanz zählt
grade_prompt = ChatPromptTemplate([
    ("system", """Rolle: strenger Relevanz-Bewerter.
Gib NUR 'yes' zurück wenn das Dokument DIREKTE Informationen zur Beantwortung enthält.
Gib 'no' zurück wenn:
- Das Dokument nur vage verwandt ist
- Die Anfrage zu unklar oder vage für dieses Dokument ist
- Keine konkreten Fakten zur Anfrage vorhanden sind"""),
    ("user", """Dokument:
{document}

Benutzeranfrage: {query}

Ist dieses Dokument direkt relevant?""")
])

# Grading Chain
grader_chain = grade_prompt | grader_llm

print(f"✅ Document Grader erstellt (strenger Modus)")
print(f"   grader_llm: with_structured_output + with_retry(stop_after_attempt={MAX_RETRIES})")

✅ Document Grader erstellt (strenger Modus)
   grader_llm: with_structured_output + with_retry(stop_after_attempt=3)


**Was passiert hier?**

1. `with_structured_output(...)` — bindet das Pydantic-Schema ans Modell und erzwingt strukturierte Ausgabe

<p><font color='black' size='5'>
Graph Nodes definieren
</font></p>

In [9]:
def retrieve_node(state: SelfRAGState) -> SelfRAGState:
    """Ruft Dokumente ab"""
    query = state["query"]
    documents = retriever.invoke(query, config={**run_cfg, "run_name": "m28_kap3_retrieve_node"})
    print(f"🔍 Retrieved {len(documents)} Dokumente")
    return {"documents": documents}

def grade_documents_node(state: SelfRAGState) -> SelfRAGState:
    """Bewertet jedes Dokument auf Relevanz"""
    query = state["query"]
    documents = state["documents"]

    relevant_docs = []
    scores = []

    for doc in documents:
        # Dokument bewerten
        grade = grader_chain.invoke({
            "document": doc.page_content,
            "query": query
        }, config=run_cfg)

        scores.append(grade.score)

        if grade.score == "yes":
            relevant_docs.append(doc)
            print(f"✅ Relevant: {doc.page_content[:50]}...")
        else:
            print(f"❌ Nicht relevant: {doc.page_content[:50]}...")

    print(f"\n📊 {len(relevant_docs)}/{len(documents)} Dokumente relevant")

    return {
        "relevant_docs": relevant_docs,
        "relevance": scores
    }

def generate_node(state: SelfRAGState) -> SelfRAGState:
    """Generiert Antwort basierend auf relevanten Dokumenten"""
    query = state["query"]
    docs = state["relevant_docs"]

    if not docs:
        return {"answer": "Nicht im Korpus."}

    # Kontext zusammenstellen
    context = "\n\n".join([doc.page_content for doc in docs])

    # RAG Prompt
    rag_prompt = ChatPromptTemplate([
        ("system", "Rolle: hilfreicher Briefing-Assistent. Antwort nur auf Basis des Kontexts formulieren."),
        ("user", """Kontext:
{context}

Frage: {query}

Antwort:""")
    ])

    # Generierung
    chain = rag_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "query": query}, config={**run_cfg, "run_name": "m28_kap3_generate_node"})

    print(f"\n🤖 Antwort generiert ({len(answer)} Zeichen)")

    return {"answer": answer}

print("✅ Nodes definiert")

✅ Nodes definiert


<p><font color='black' size='5'>
Graph aufbauen
</font></p>

In [ ]:
# Self-RAG Workflow erstellen
self_rag_builder = StateGraph(SelfRAGState)

# Nodes hinzufügen
self_rag_builder.add_node("retrieve", retrieve_node)
self_rag_builder.add_node("grade_documents", grade_documents_node)
self_rag_builder.add_node("generate", generate_node)

# Edges definieren
self_rag_builder.add_edge(START, "retrieve")
self_rag_builder.add_edge("retrieve", "grade_documents")
self_rag_builder.add_edge("grade_documents", "generate")
self_rag_builder.add_edge("generate", END)

# Graph kompilieren
self_rag_agent = self_rag_builder.compile()

print("✅ Self-RAG Graph kompiliert")

<p><font color='black' size='5'>
Graph Visualisierung
</font></p>

In [ ]:
# --- VIZ ...
from IPython.display import Image, display
display(Image(self_rag_agent.get_graph().draw_mermaid_png()))

<p><font color='black' size='5'>
Self-RAG testen
</font></p>

In [ ]:
# Test Query
test_query = "Wie unterscheidet sich naive RAG von advanced oder modular RAG?"

# Workflow ausführen
self_rag_result = self_rag_agent.invoke(
    {"query": test_query},
    config={"run_name": "m28_kap3_selfrag_demo", "tags": ["m28", "self-rag"]},
)

# Ergebnis anzeigen
mprint("## 🎯 Self-RAG Ergebnis")
mprint("---")
mprint(f"**Query:** {test_query}")
mprint(f"**Antwort:** {self_rag_result['answer']}")

# 4 | Reranking: Dokumente neu ordnen
---

<p><font color='black' size='5'>
Konzept
</font></p>

**Vektor-Similarity ist ein schnelles, aber grobes Relevanz-Mass.**
Die Top-K Dokumente sind nach Cosine-Ähnlichkeit sortiert –
das muss nicht die beste Reihenfolge für die Antwort sein.

**Reranking** bewertet die abgerufenen Dokumente ein zweites Mal,
präziser und kontextbezogener:

```
Query → Retrieve (k=10) → [RERANK] → Top-3 → Generate
```

| Schritt | Methode | Geschwindigkeit | Genauigkeit |
|---------|---------|----------------|-------------|
| Retrieval | Cosine-Similarity | ✅ Sehr schnell | ⚠️ Grob |
| Reranking | CrossEncoder / LLM | ⚠️ Langsamer | ✅ Präzise |

**Zwei Ansätze:**
- **LLM-Reranker:** Gibt Relevanz-Score 0–10, keine zusätzlichen Pakete
- **CrossEncoder:** HuggingFace-Modell, schneller als LLM, sehr präzise

Wir implementieren den **LLM-Reranker** (einfach, direkt nutzbar).

In [13]:
#@markdown   <p><font size="4" color='green'>  RAG mit Reranking</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    Q[/Query/] --> RET["Retrieve\nk=6 Docs"]
    RET --> RR["Reranker\nScore 0-10"]
    RR --> TOP["Top-3\n(höchste Scores)"]
    TOP --> GEN[Generate]
    GEN --> ANT[/Antwort/]

    subgraph STANDARD["Standard RAG"]
        RET2["Retrieve k=3"] --> GEN2[Generate]
    end

    style RR  fill:#FF9800,color:#fff
    style TOP fill:#4CAF50,color:#fff
    style STANDARD fill:#F5F5F5
'''
mermaid(diagram, width=800)

<p><font color='black' size='5'>
LLM-Reranker implementieren
</font></p>

Der Reranker bewertet jedes Dokument mit einem Score von 0–10
und sortiert die Dokumente nach Relevanz.

In [14]:
class RelevanceScoreResult(BaseModel):
    """Bewertet die Relevanz eines Dokuments für eine Query."""
    score: int = Field(
        description="Relevanz-Score von 0 (irrelevant) bis 10 (perfekt relevant)",
        ge=0, le=10
    )
    begruendung: str = Field(description="Kurze Begründung für den Score")

# .with_retry(): schützt vor transienten API-Fehlern beim Reranking
reranker_llm = llm.with_structured_output(RelevanceScoreResult).with_retry(stop_after_attempt=MAX_RETRIES)

rerank_prompt = ChatPromptTemplate([
    ("system",
     "Rolle: Relevanzbewertung für Dokumente und Suchanfragen.\n"
     "Score 0: Komplett irrelevant.\n"
     "Score 5: Teilweise relevant, aber lückenhaft.\n"
     "Score 10: Direkt und vollständig relevant."),
    ("user",
     "Suchanfrage: {query}\n\nDokument:\n{document}\n\nRelevanz-Score (0-10):")
])

reranker_chain = rerank_prompt | reranker_llm

def rerank_documents(query: str, docs: list, top_k: int = K_RETRIEVAL) -> list:
    """Bewertet und sortiert Dokumente nach Relevanz-Score."""
    scored = []
    for doc in docs:
        result = reranker_chain.invoke({
            "query": query,
            "document": doc.page_content[:600]
        }, config=run_cfg)
        scored.append((doc, result.score, result.begruendung))
        print(f"  Score {result.score}/10: {doc.page_content[:60]}...")

    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, score, _ in scored[:top_k]], scored

print(f"Reranker definiert")
print(f"   reranker_llm: with_structured_output + with_retry(stop_after_attempt={MAX_RETRIES})")
print(f"   Default top_k = K_RETRIEVAL = {K_RETRIEVAL}")

Reranker definiert
   reranker_llm: with_structured_output + with_retry(stop_after_attempt=3)
   Default top_k = K_RETRIEVAL = 3


**Was passiert hier?**

1. `with_structured_output(...)` — bindet das Pydantic-Schema ans Modell und erzwingt strukturierte Ausgabe

<p><font color='black' size='5'>
Reranking in Self-RAG integrieren
</font></p>

Wir erweitern die Self-RAG Pipeline um einen Reranking-Schritt
zwischen Retrieve und Grade:

In [15]:
class RerankState(TypedDict):
    query:              str
    documents:          List[Document]   # Initiales Retrieval (K_WIDE)
    reranked_docs: List[Document]   # Nach Reranking (Top-K_RETRIEVAL)
    relevant_docs: List[Document]   # Nach Grading
    answer:             str
    relevance:   List[str]

# K_WIDE: mehr Kandidaten für Reranking → bessere Auswahl
retriever_wide = vectorstore.as_retriever(search_kwargs={"k": K_WIDE})

def retrieve_wide_node(state: RerankState) -> dict:
    docs = retriever_wide.invoke(state["query"], config={**run_cfg, "run_name": "m28_kap4_retrieve_wide_node"})
    print(f"Retrieved {len(docs)} Docs (wide, k={K_WIDE})")
    return {"documents": docs}

def rerank_node(state: RerankState) -> dict:
    """Rerankt die Dokumente und gibt Top-K_RETRIEVAL zurück."""
    print(f"Reranking {len(state['documents'])} Dokumente...")
    top_docs, scored = rerank_documents(state["query"], state["documents"], top_k=K_RETRIEVAL)
    print(f"Top-{K_RETRIEVAL} nach Reranking:")
    for doc, score, reason in scored[:K_RETRIEVAL]:
        print(f"  {score}/10 | {reason[:60]}")
    return {"reranked_docs": top_docs}

def grade_reranked_node(state: RerankState) -> dict:
    """Gradet nur die Top reranked Dokumente."""
    query = state["query"]
    relevant_docs, scores = [], []
    for doc in state["reranked_docs"]:
        grade = grader_chain.invoke({"document": doc.page_content, "query": query}, config=run_cfg)
        scores.append(grade.score)
        if grade.score == "yes":
            relevant_docs.append(doc)
    print(f"{len(relevant_docs)}/{len(state['reranked_docs'])} Docs nach Grading relevant")
    return {"relevant_docs": relevant_docs, "relevance": scores}

def generate_rerank_node(state: RerankState) -> dict:
    docs = state["relevant_docs"] or state["reranked_docs"]
    fallback_hinweis = "" if state["relevant_docs"] else "Hinweis: Grading hat keine Dokumente freigegeben; die Antwort nutzt die Top-Reranking-Kandidaten.\n"
    context = "\n\n".join(
        f"Quelle: {d.metadata.get('source', 'unbekannt')}\n{d.page_content}"
        for d in docs
    )
    prompt = ChatPromptTemplate([
        ("system", "Antwort ausschließlich aus dem Kontext ableiten. Wenn die gesuchten Fakten fehlen: 'Nicht im Korpus' schreiben. Deutsch."),
        ("user", "{fallback_hinweis}Kontext:\n{context}\n\nFrage: {query}\n\nAntwort mit Quellenhinweis:")
    ])
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "fallback_hinweis": fallback_hinweis,
        "context": context,
        "query": state["query"],
    }, config={**run_cfg, "run_name": "m28_kap4_generate_rerank_node"})
    return {"answer": answer}

print(f"Rerank-RAG Nodes definiert")
print(f"   Retrieve: k={K_WIDE} → Rerank → Top-{K_RETRIEVAL} → Grade → Generate")

Rerank-RAG Nodes definiert
   Retrieve: k=6 → Rerank → Top-3 → Grade → Generate


In [ ]:
rerank_builder = StateGraph(RerankState)
rerank_builder.add_node("retrieve",  retrieve_wide_node)
rerank_builder.add_node("rerank",    rerank_node)
rerank_builder.add_node("grade",     grade_reranked_node)
rerank_builder.add_node("generate",  generate_rerank_node)

rerank_builder.add_edge(START,      "retrieve")
rerank_builder.add_edge("retrieve", "rerank")
rerank_builder.add_edge("rerank",   "grade")
rerank_builder.add_edge("grade",    "generate")
rerank_builder.add_edge("generate", END)

rerank_agent = rerank_builder.compile()
print("Rerank-RAG Graph kompiliert")

try:
    display(Image(rerank_agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph-Visualisierung: {e}")

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_edge(...)` — verbindet zwei Knoten mit einer festen Kante
4. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

<p><font color='black' size='5'>
Reranking testen
</font></p>

In [ ]:
query_rr = "Welche vier Phasen verwendet der Survey von Huang und Huang zur Strukturierung von RAG?"

mprint("## Reranking: Scoring der Kandidaten")
result_rr = rerank_agent.invoke(
    {"query": query_rr},
    config={"run_name": "m28_kap4_rerankrag_demo", "tags": ["m28", "rerank-rag"]},
)

mprint("---")
mprint(f"**Query:** {query_rr}")
mprint(f"**Relevante Docs nach Reranking+Grading:** {len(result_rr['relevant_docs'])}")
mprint(f"**Antwort:** {result_rr['answer']}")

# 5 | Corrective RAG: Query Rewriting
---

<p><font color='black' size='5'>
Konzept
</font></p>

**Corrective RAG (CRAG) erkennt schlechte Retrievals und reagiert:**

```
Query → Retrieve → Grade
                      ↓
            Gut? → Generate
                      ↓
         Schlecht? → Rewrite Query → Retrieve wieder
```

**Warum wichtig?**
- Nutzer-Queries sind oft ungenau oder missverständlich
- Durch Umformulierung werden bessere Dokumente gefunden
- Erhöht Erfolgsrate drastisch

<p><font color='black' size='5'>
State Definition
</font></p>

In [18]:
class CorrectiveRAGState(TypedDict):
    """State für Corrective RAG"""
    initial_query: str           # Ursprüngliche Query
    query: str                    # Aktuelle Query (kann umgeschrieben sein)
    documents: List[Document]
    relevant_docs: List[Document]
    answer: str
    attempts: int              # Wie oft wurde umgeschrieben?
    has_relevant: bool       # Flag für Routing

<p><font color='black' size='5'>
Query Rewriter
</font></p>

In [19]:
# Query Rewriting Prompt
rewrite_prompt = ChatPromptTemplate([
    ("system", "Rolle: Experte für das Umformulieren von Suchanfragen."),
    ("user", """Die folgende Suchanfrage hat keine guten Ergebnisse geliefert.
Die Anfrage so umformulieren, dass bessere Dokumente gefunden werden.
Spezifischer formulieren und passende Synonyme nutzen.

Ursprüngliche Query: {query}

Verbesserte Query:""")
])

# Rewriting Chain
rewriter_chain = rewrite_prompt | llm | StrOutputParser()

print("✅ Query Rewriter erstellt")

✅ Query Rewriter erstellt


<p><font color='black' size='5'>
CRAG Nodes
</font></p>

In [20]:
def crag_retrieve_node(state: CorrectiveRAGState) -> CorrectiveRAGState:
    """Retrieval für CRAG"""
    query = state["query"]
    documents = retriever.invoke(query, config={**run_cfg, "run_name": "m28_kap5_crag_retrieve_node"})
    print(f"🔍 Retrieved {len(documents)} Dokumente für Query: '{query}'")
    return {"documents": documents}

def crag_grade_node(state: CorrectiveRAGState) -> CorrectiveRAGState:
    """Grading für CRAG mit Entscheidungslogik"""
    query = state["query"]
    documents = state["documents"]

    relevant_docs = []

    for doc in documents:
        grade = grader_chain.invoke({
            "document": doc.page_content,
            "query": query
        }, config=run_cfg)

        if grade.score == "yes":
            relevant_docs.append(doc)

    # Entscheidung: Gibt es genug relevante Dokumente?
    has_relevant = len(relevant_docs) > 0

    if has_relevant:
        print(f"✅ {len(relevant_docs)} relevante Dokumente gefunden")
    else:
        print(f"❌ Keine relevanten Dokumente → Query wird umgeschrieben")

    return {
        "relevant_docs": relevant_docs,
        "has_relevant": has_relevant
    }

def rewrite_query_node(state: CorrectiveRAGState) -> CorrectiveRAGState:
    """Schreibt Query um"""
    original = state["initial_query"]
    current = state["query"]
    retry = state.get("attempts", 0)

    # Query umschreiben
    new_query = rewriter_chain.invoke({"query": current}, config={**run_cfg, "run_name": "m28_kap5_rewrite_query_node"})

    print(f"🔄 Query umgeschrieben (Versuch {retry + 1})")
    print(f"   Alt: '{current}'")
    print(f"   Neu: '{new_query}'")

    return {
        "query": new_query,
        "attempts": retry + 1
    }

def crag_generate_node(state: CorrectiveRAGState) -> CorrectiveRAGState:
    """Generierung für CRAG mit expliziter Quellenbindung."""
    query = state["initial_query"]
    docs = state["relevant_docs"]
    if not docs:
        return {"answer": "Nicht im Korpus."}

    context = "\n\n".join(
        f"Quelle: {doc.metadata.get('source', 'unbekannt')}\n{doc.page_content}"
        for doc in docs
    )

    rag_prompt = ChatPromptTemplate([
        ("system", "Antwort ausschließlich aus dem Kontext ableiten. Wenn die gesuchten Fakten fehlen: 'Nicht im Korpus' schreiben. Deutsch."),
        ("user", "Kontext:\n{context}\n\nFrage: {query}\n\nAntwort mit Quellenhinweis:")
    ])

    chain = rag_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "query": query}, config={**run_cfg, "run_name": "m28_kap5_crag_generate_node"})

    return {"answer": answer}

print("✅ CRAG Nodes definiert")

✅ CRAG Nodes definiert


<p><font color='black' size='5'>
CRAG Graph mit Conditional Routing
</font></p>

In [ ]:
# Routing Funktion
def grading_router(state: CorrectiveRAGState) -> Literal["generate", "rewrite_query"]:
    """Entscheidet: Generate oder Rewrite?"""
    attempts = state.get("attempts", 0)
    has_relevant = state["has_relevant"]

    # Maximal 2 Rewrite-Versuche
    if has_relevant or attempts >= 2:
        return "generate"
    else:
        return "rewrite_query"

# CRAG Workflow
crag_builder = StateGraph(CorrectiveRAGState)

# Nodes
crag_builder.add_node("retrieve", crag_retrieve_node)
crag_builder.add_node("grade", crag_grade_node)
crag_builder.add_node("rewrite_query", rewrite_query_node)
crag_builder.add_node("generate", crag_generate_node)

# Edges
crag_builder.add_edge(START, "retrieve")
crag_builder.add_edge("retrieve", "grade")

# Conditional Edge: Nach Grading
crag_builder.add_conditional_edges(
    "grade",
    grading_router,
    {
        "generate": "generate",
        "rewrite_query": "rewrite_query"
    }
)

# Nach Rewrite wieder zu Retrieve
crag_builder.add_edge("rewrite_query", "retrieve")
crag_builder.add_edge("generate", END)

# Kompilieren
crag_agent = crag_builder.compile()

print("✅ CRAG Graph mit Conditional Routing kompiliert")

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_conditional_edges(...)` — legt bedingte Übergänge zwischen Knoten fest
4. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

<p><font color='black' size='5'>
CRAG Graph Visualisierung
</font></p>

In [ ]:
# --- VIZ ---
display(Image(crag_agent.get_graph().draw_mermaid_png()))

<p><font color='black' size='5'>
CRAG testen
</font></p>

In [ ]:
# Test mit einer Query, die nicht im Datensatz ist → Rewrite soll auslösen
test_query = "Welche praktischen Probleme löst Query Rewriting in RAG-Systemen?"

crag_result = crag_agent.invoke(
    {
        "initial_query": test_query,
        "query": test_query,
        "attempts": 0,
    },
    config={"run_name": "m28_kap5_crag_demo", "tags": ["m28", "crag"]},
)

mprint("## 🎯 CRAG Ergebnis")
mprint("---")
mprint(f"**Ursprüngliche Query:** {crag_result['initial_query']}")
mprint(f"**Finale Query:** {crag_result['query']}")
mprint(f"**Rewrite-Versuche:** {crag_result.get('attempts', 0)}")
mprint(f"**Antwort:** {crag_result['answer']}")

# 6 | Vergleich: Basic vs. Advanced RAG
---

| Aspekt                  | Basic RAG              | Self-RAG                     | Corrective RAG                 | Reranking                                  |
| ----------------------- | ---------------------------- | ---------------------------- | ------------------------------ | ------------------------------------------ |
| **Workflow**            | Linear (Retrieve → Generate) | Mit Self-Grading             | Mit Retry-Loop + Query-Rewrite | Retrieve → Score → Sort                    |
| **Dokument-Qualität**   | Nicht geprüft                | ✅ Geprüft (LLM-basiert)      | ✅ Geprüft (mit Feedback-Loop)  | ⚠️ Indirekt verbessert (durch Sortierung)  |
| **Query-Verbesserung**  | ❌ Nein                       | ❌ Nein                       | ✅ Ja (Query-Rewriting)         | ❌ Nein                                     |
| **Anpassungsfähigkeit** | ❌ Statisch                   | ⚠️ Teilweise adaptiv         | ✅ Dynamisch adaptiv            | ❌ Statisch                                 |
| **Komplexität**         | Niedrig                      | Mittel                       | Hoch                           | Mittel                                     |
| **Reranking**           | ❌ Nein                       | ❌ Nein                       | ❌ Nein (optional integrierbar) | ✅ Kernbestandteil                          |
| **Use Case**            | Einfache Q&A                 | Qualitätskritische Antworten | Ambige/unklare Queries         | Qualitätssteigerung bestehender Retrievals |



**Empfehlung:**
- **Basic RAG:** Für Prototyping und klare Use Cases
- **Self-RAG:** Wenn Retrieval-Qualität wichtig ist
- **Corrective RAG:** Wenn Nutzer-Queries ungenau sind

# 7 | Best Practices & Tipps
---

<p><font color='black' size='5'>
Performance-Optimierung
</font></p>

**Grading ist teuer (LLM-Calls):**
- ✅ Cache Grading-Ergebnisse für identische Dokumente
- ✅ Paralleles Grading mit `asyncio` (für große Dokumentenmengen)
- ✅ Alternative: Lightweight Relevance Scorer (z.B. Cosine-Similarity-Threshold)

**Rewriting Limits:**
- ✅ Maximal 2-3 Rewrite-Versuche (sonst zu langsam)
- ✅ Fallback: Web-Search wenn kein Doc gefunden wird

**Monitoring:**
- ✅ Logge Rewrite-Rate (zu hoch → Retrieval-Problem)
- ✅ Tracke Relevance-Scores (zu niedrig → Daten-Problem)

<p><font color='black' size='5'>
Wann welches Pattern?
</font></p>

| Szenario | Empfohlenes Pattern |
|----------|---------------------|
| **Hohe Datenqualität, klare Queries** | Basic RAG |
| **Viele irrelevante Retrieval-Ergebnisse** | Self-RAG |
| **Nutzer mit ungenauen Fragen** | Corrective RAG |
| **Kritische Anwendung (Legal, Medical)** | Self-RAG + Hallucination Check |
| **Niedrige Latenz wichtig** | Basic RAG (schneller) |
| **Hohe Accuracy wichtig** | Corrective RAG (besser) |

In [24]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("m28-advanced-rag", limit=3, show_steps=True)

## LangSmith Trace — `M28-Advanced-RAG`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `M28-CRAG-Demo` | ✅ success | 4.3s | 0 |
| `M28-RerankRAG-Demo` | ✅ success | 12.7s | 0 |
| `M28-SelfRAG-Demo` | ✅ success | 8.1s | 0 |


### Steps — letzter Run: `M28-CRAG-Demo`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `chain` | `generate` | ✅ | 1.9s |
| 2 | `chain` | `grade` | ✅ | 2.2s |
| 3 | `chain` | `retrieve` | ✅ | 0.2s |

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf im Kurs als Lernunterstützung genutzt werden, z. B. um Fehlermeldungen zu verstehen, Teilschritte zu klären oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size='5'> Aufgabe 1: Grounding-Prüfung für Self-RAG </font></p>

Erweitere den vorhandenen Self-RAG-Ablauf nach generate_node um eine
Grounding-Prüfung:

1. Definiere ein Pydantic-Modell GroundingAssessment mit
   is_grounded: bool und explanation: str.
2. Prüfe die Antwort gegen relevant_documents mit
   with_structured_output(GroundingAssessment).
3. Bei is_grounded == False darf höchstens einmal neu generiert werden;
   verwende dabei ausschließlich die relevanten Dokumente.
4. Wenn auch der zweite Versuch unbelegt bleibt, gib exakt
   Nicht im Korpus. zurück.

Nutze SelfRAGState und die vorhandenen Nodes als Ausgangspunkt. Ein neues
Bewertungs-Framework ist nicht erforderlich.


<p><font color='black' size='5'>
Aufgabe 2: Reranking vergleichen
</font></p>

Vergleiche Retrieval mit und ohne Reranking:

1. Für drei Testfragen die Top-3-Dokumente der Basis-Suche und nach dem Reranking ausgeben.
2. Prüfen, bei welchen Fragen sich die Reihenfolge ändert.
3. Kurz bewerten, ob das Reranking die Antwortqualität verbessert.

**Tipp:** Die Reranking-Funktion aus dem Reranking-Kapitel wiederverwenden.


<p><font color='black' size='5'> Aufgabe 3: Adaptive RAG </font></p>

Implementiere einen Classifier, der eine Anfrage als simple, complex oder
ambiguous einordnet, und teste das Routing:

- simple: eine kleine Funktion basic_rag_answer(query), die den vorhandenen
  Retriever einmal aufruft und den bestehenden Generierungs-Prompt verwendet,
- complex: der vorhandene self_rag_agent,
- ambiguous: der vorhandene crag_agent.

Teste mindestens eine Anfrage pro Kategorie. Ein eigener Basic-RAG-Graph ist
nicht erforderlich; die einfache Funktion genügt. Optional kann der Classifier
mit with_structured_output() umgesetzt werden.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [RAG-Pipeline](https://editor.p5js.org/ralf.bendig.rb/full/RrfB3nCwK)
- [Embeddings 2D](https://editor.p5js.org/ralf.bendig.rb/full/LPjLkzWbE)
- [Embeddings 3D](https://editor.p5js.org/ralf.bendig.rb/full/gFBwB2E8n)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)
- [Long Context & CAG](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/long-context-cag.html)
- [Embeddings](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/embeddings.html)
- [Einsteiger ChromaDB](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-chromadb.html)
- [Agent Evaluation & Observability Best Practices](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/agent-evaluation-observability-best-practices.html)
